In [1]:
library(tidyverse)
library(lubridate)
library(readxl)

Warning message:
"package 'tidyverse' was built under R version 3.6.3"-- Attaching packages --------------------------------------- tidyverse 1.3.2 --v ggplot2 3.4.0      v purrr   1.0.0 
v tibble  3.1.8      v dplyr   1.0.10
v tidyr   1.2.1      v stringr 1.4.0 
v readr   2.1.3      v forcats 0.5.2 Warning message:
"package 'ggplot2' was built under R version 3.6.3"Warning message:
"package 'tibble' was built under R version 3.6.3"Warning message:
"package 'tidyr' was built under R version 3.6.3"Warning message:
"package 'readr' was built under R version 3.6.3"Warning message:
"package 'purrr' was built under R version 3.6.3"Warning message:
"package 'dplyr' was built under R version 3.6.3"Warning message:
"package 'forcats' was built under R version 3.6.3"-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()Warning message:
"package 'lubridate' was built under R version 3.6.3"Loadi

In [2]:
#loading all the dataset 
fertilizer    <- read_excel("E:/Users/BROTHER COMPUTER/Desktop/research/Raw_Data/(fertilizer)CMO-Historical-Data-Monthly.xlsx", sheet = "Monthly Prices")
CPI           <- read.csv("E:/Users/BROTHER COMPUTER/Desktop/research/Raw_Data/CPI_DATA.csv")
crude_oil     <- read.csv("E:/Users/BROTHER COMPUTER/Desktop/research/Raw_Data/crude oil data.csv")
FX            <- read.csv("E:/Users/BROTHER COMPUTER/Desktop/research/Raw_Data/Exchange_Rate.csv")
fuel          <- read_excel("E:/Users/BROTHER COMPUTER/Desktop/research/Raw_Data/Fuel_Prices.xlsx")



In [3]:
#standardising the date into a consistent format 
fertilizer <- mutate(fertilizer , Date = ym(Date))

CPI <- mutate(CPI, 
              start_month = str_extract(Western.Timeline, "(?<=Mid-)[A-Za-z]{3}"),
                date_str = paste("15",start_month,Year),
                 Date = dmy(date_str))
crude_oil <- mutate(crude_oil , Date = ymd(observation_date))
FX <- mutate(FX, Date= mdy(Date))
fuel <- mutate(fuel, Date = ymd(`Effective Date (AD)`))



In [4]:
fertilizer <- fertilizer %>% slice(-1)

brent_monthly <- crude_oil %>%
  mutate(Date = floor_date(Date, "month")) %>%
  group_by(Date) %>%
  summarise(brent = mean(DCOILBRENTEU, na.rm = TRUE))

In [5]:
#filtering the dataframe and choosing only required column
refined_fert <- select(fertilizer, Date, DAP, Urea)
refined_CPI <- select(CPI, Date,inflation_rate = `Y.o.Y.Inflation.Rate....`) %>%
    mutate(Date = floor_date(Date , "month"))

refined_crude_oil <- select( brent_monthly, Date, oil_price = brent)
refined_FX <- select(FX , Date, exchange_rate = Price)
refined_fuel<- select(fuel, Date, Diesel)



In [6]:
study_start <- as.Date("2013-07-01")
study_end   <- as.Date("2025-12-01")
df_list <- list(refined_fert,refined_CPI,refined_crude_oil,refined_FX)
macro_clean <- map(df_list, ~ filter(.x, Date >= study_start & Date <= study_end))
macro_csv <- reduce(macro_clean, full_join, by = "Date")
macro_csv <- arrange(macro_csv, Date)

In [7]:
write.csv(macro_csv,"E:/Users/BROTHER COMPUTER/Desktop/research/Cleaned_Data/macro_clean.csv")

In [8]:
write.csv(macro_csv,"E:/Users/BROTHER COMPUTER/Desktop/research/Cleaned_Data/diesel_clean.csv")